# L7: 多模型抽象设计

> 深入理解工厂模式，学习如何设计可扩展的多模型抽象层

In [ ]:
# === 环境设置 ===
import sys
import os
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

# 验证模块导入
try:
    from backend.llm import LLMFactory, BaseLLM
    print("✓ LLM 模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

## 7.1 为什么需要多模型抽象？

### 问题场景

```
场景 1: 开发环境用 Ollama（免费本地）
        生产环境用 OpenAI（稳定可靠）

场景 2: 简单问答用 DeepSeek（便宜）
        复杂推理用 GPT-4（强大）

场景 3: 国内用 DeepSeek/GLM
        国外用 OpenAI/Anthropic
```

### 没有抽象的痛点

| 痛点 | 说明 |
|------|------|
| 代码重复 | 每个模型写一遍调用逻辑 |
| 难以切换 | 换模型要改很多代码 |
| 难以扩展 | 加新模型要改现有代码 |
| 维护困难 | 多处相似代码，bug 修复麻烦 |

## 7.2 工厂模式回顾

### 工厂模式结构

```
         LLMFactory
              │
              ├── create(provider) → BaseLLM
              │
              └── _providers = {
                     "openai": OpenAILLM,
                     "deepseek": DeepSeekLLM,
                     "anthropic": AnthropicLLM,
                     ...
                  }
```

In [ ]:
from backend.llm.factory import LLMFactory

# 查看工厂的内部结构
print("=== LLMFactory 内部结构 ===\n")
print(f"已注册提供商: {LLMFactory.list_providers()}")
print(f"\n提供商字典: {LLMFactory._providers}")

## 7.3 抽象基类：BaseLLM

### 模板方法模式

BaseLLM 定义了所有 LLM 客户端必须实现的接口：

In [ ]:
from backend.llm.base import BaseLLM
import inspect

print("=== BaseLLM 抽象方法 ===\n")

# 获取所有抽象方法
abstract_methods = [
    name for name, method in inspect.getmembers(BaseLLM, predicate=inspect.isfunction)
    if hasattr(method, '__isabstractmethod__') and method.__isabstractmethod__
]

for method_name in abstract_methods:
    method = getattr(BaseLLM, method_name)
    sig = inspect.signature(method)
    print(f"  {method_name}{sig}")

print("\n=== BaseLLM 公共方法 ===\n")

# 获取公共方法
public_methods = [
    name for name in dir(BaseLLM)
    if not name.startswith('_') and callable(getattr(BaseLLM, name))
]

for method_name in ['supports_tools', 'supports_embedding', 'get_model_info']:
    if method_name in public_methods:
        method = getattr(BaseLLM, method_name)
        sig = inspect.signature(method)
        print(f"  {method_name}{sig}")

### BaseLLM 核心设计

In [ ]:
# 查看部分源码
import inspect

print("=== BaseLLM.__init__ 源码 ===\n")
print(inspect.getsource(BaseLLM.__init__))

print("\n=== BaseLLM.chat 源码（签名）===\n")
print(inspect.signature(BaseLLM.chat))

## 7.4 具体实现对比

### 不同提供商的实现差异

In [ ]:
from backend.llm.deepseek_client import DeepSeekLLM
from backend.llm.openai_client import OpenAILLM
from backend.llm.anthropic_client import AnthropicLLM

# 对比不同实现
providers = [
    ("DeepSeek", DeepSeekLLM),
    ("OpenAI", OpenAILLM),
    ("Anthropic", AnthropicLLM),
]

print("=== 不同 LLM 实现对比 ===\n")
print(f"{'提供商':<15} {'支持工具':<10} {'支持嵌入':<10}")
print("-" * 40)

for name, cls in providers:
    # 创建临时实例检查
    # 实际使用需要 API key
    print(f"{name:<15} {'✓':<10} {'✗':<10}")

print("\n注意: Anthropic 支持工具，OpenAI 支持工具和嵌入")

### API 格式差异处理

In [ ]:
print("=== 各提供商 API 差异 ===\n")

differences = [
    ("DeepSeek", "OpenAI 兼容", "chat.completions.create", "与 OpenAI 相同"),
    ("OpenAI", "原生 OpenAI", "chat.completions.create", "标准格式"),
    ("Anthropic", "Messages API", "messages.create", "不同格式"),
    ("Ollama", "REST API", "chat", "自定义格式"),
    ("GLM", "智谱 API", "chat/completions", "类 OpenAI"),
]

print(f"{'提供商':<15} {'API 类型':<20} {'端点':<30} {'备注'}")
print("-" * 80)
for row in differences:
    print(f"{row[0]:<15} {row[1]:<20} {row[2]:<30} {row[3]}")

print("\n结论: 各提供商 API 格式不同，需要适配层统一接口")

## 7.5 数据模型统一

### Pydantic 模型的作用

不同提供商返回不同格式的数据，需要统一的内部格式：

In [ ]:
from backend.llm.models import Message, ChatResponse, TokenUsage, ToolCall

print("=== 统一数据模型 ===\n")

# Message
print("Message 模型:")
msg = Message(role="user", content="你好")
print(f"  {msg.model_dump_json(indent=2)}\n")

# ChatResponse
print("ChatResponse 结构:")
response = ChatResponse(
    content="你好！",
    role="assistant",
    usage=TokenUsage(prompt_tokens=10, completion_tokens=5, total_tokens=15)
)
print(f"  {response.model_dump_json(indent=2)}\n")

print("优势: 所有提供商返回相同的数据结构，便于上层处理")

## 7.6 扩展：添加新的 LLM 提供商

### 步骤演示

假设要添加一个新的 LLM 提供商 `MyLLM`：

In [ ]:
# 示例：如何添加新的 LLM 提供商

template = '''
# 1. 在 backend/llm/myllm_client.py 创建客户端类

from typing import List, Optional
from backend.llm.base import BaseLLM
from backend.llm.models import ChatResponse, Message, ToolDefinition

class MyLLM(BaseLLM):
    """MyLLM 客户端实现"""
    
    def __init__(self, model: str, api_key: str, base_url: str = "https://api.myllm.com", **kwargs):
        super().__init__(model=model, api_key=api_key, base_url=base_url, **kwargs)
        # 初始化 MyLLM 特定的客户端
    
    async def chat(
        self,
        messages: List[Message],
        tools: Optional[List[ToolDefinition]] = None,
        temperature: float = 0.7,
        max_tokens: Optional[int] = None,
        **kwargs
    ) -> ChatResponse:
        """实现 chat 方法"""
        # 1. 转换消息格式为 MyLLM API 格式
        # 2. 调用 MyLLM API
        # 3. 转换响应为统一的 ChatResponse 格式
        pass
    
    async def embed(self, texts: List[str], **kwargs):
        """实现 embed 方法（如果支持）"""
        pass
    
    def supports_tools(self) -> bool:
        return True
    
    def supports_embedding(self) -> bool:
        return False

# 2. 在 backend/llm/factory.py 注册

from backend.llm.myllm_client import MyLLM

class LLMFactory:
    _providers = {
        "openai": OpenAILLM,
        "deepseek": DeepSeekLLM,
        "myllm": MyLLM,  # 添加新提供商
    }

# 3. 使用

llm = LLMFactory.create("myllm", api_key="your-key")
response = await llm.chat([Message(role="user", content="你好")])
'''

print(template)

## 7.7 配置管理

In [ ]:
# 查看项目配置
from pathlib import Path

env_file = Path(project_path) / ".env"
if env_file.exists():
    print("=== .env 配置示例 ===\n")
    print("# LLM API Keys")
    print("DEEPSEEK_API_KEY=your_deepseek_key")
    print("OPENAI_API_KEY=your_openai_key")
    print("ANTHROPIC_API_KEY=your_anthropic_key")
    print("GLM_API_KEY=your_glm_key")
    print("\n# 默认 LLM 提供商")
    print("DEFAULT_LLM_PROVIDER=deepseek")
    print("\n# 模型配置")
    print("DEEPSEEK_MODEL=deepseek-chat")
    print("OPENAI_MODEL=gpt-4o-mini")
else:
    print(".env 文件不存在")

## 7.8 综合练习

### 练习：动态切换 LLM 提供商

In [ ]:
class LLMRouter:
    """LLM 路由器 - 根据任务类型选择合适的模型"""
    
    def __init__(self):
        self.providers = {}
    
    def register_provider(self, name: str, llm_instance):
        """注册 LLM 实例"""
        self.providers[name] = llm_instance
    
    def get_provider(self, task_type: str) -> str:
        """根据任务类型返回提供商名称"""
        routing_rules = {
            "simple": "deepseek",      # 简单任务用便宜模型
            "coding": "deepseek-coder", # 代码任务用代码模型
            "complex": "openai",       # 复杂任务用 GPT-4
            "creative": "anthropic",   # 创意任务用 Claude
        }
        return routing_rules.get(task_type, "deepseek")
    
    async def process(self, task_type: str, messages: list, **kwargs):
        """处理任务"""
        provider_name = self.get_provider(task_type)
        llm = self.providers.get(provider_name)
        
        if not llm:
            raise ValueError(f"Provider {provider_name} not registered")
        
        print(f"[路由] 任务类型: {task_type} → 使用: {provider_name}")
        return await llm.chat(messages, **kwargs)

# 使用示例
router = LLMRouter()

print("=== LLM 路由器演示 ===\n")
print("任务类型路由规则:")
for task_type in ["simple", "coding", "complex", "creative"]:
    provider = router.get_provider(task_type)
    print(f"  {task_type:<10} → {provider}")

## 7.9 常见面试问题

**Q: 为什么要用工厂模式而不是直接实例化？**
- A:
  | 工厂模式 | 直接实例化 |
  |----------|------------|
  | 解耦 | 耦合具体类 |
  | 易扩展 | 改动大 |
  | 统一接口 | 接口不统一 |
  | 配置驱动 | 硬编码 |

**Q: 抽象基类（BaseLLM）的作用是什么？**
- A:
  1. 定义统一接口：所有实现类遵循相同契约
  2. 代码复用：公共逻辑在基类实现
  3. 类型提示：IDE 可以正确识别类型
  4. 强制实现：抽象方法必须被实现

**Q: 如何处理不同提供商的 API 格式差异？**
- A:
  1. 内部统一格式：使用 Pydantic 模型
  2. 适配器模式：各实现类负责格式转换
  3. 转换方法：`_convert_messages()`, `_parse_response()`

**Q: 如何选择合适的 LLM 提供商？**
- A:
  | 场景 | 推荐提供商 |
  |------|------------|
  | 成本敏感 | DeepSeek, Ollama |
  | 质量优先 | OpenAI GPT-4, Anthropic Claude |
  | 国内部署 | DeepSeek, GLM |
  | 离线使用 | Ollama |
  | 代码任务 | DeepSeek-Coder, GPT-4 |

**Q: 什么是依赖注入？项目中如何使用？**
- A:
  依赖注入是一种设计模式，将依赖对象从外部传入而非内部创建：
  ```python
  # 不好的做法
  class Agent:
      def __init__(self):
          self.llm = LLMFactory.create("deepseek")  # 硬编码
  
  # 好的做法
  class Agent:
      def __init__(self, llm: BaseLLM):  # 依赖注入
          self.llm = llm
  ```

**Q: 如何测试多模型抽象？**
- A:
  1. 单元测试：Mock LLM 响应
  2. 集成测试：使用真实 API
  3. 一致性测试：验证不同提供商返回相同结构
  4. 性能测试：对比各提供商的响应时间

---

## 总结

本课程学习了多模型抽象设计的核心知识：

| 知识点 | 说明 |
|--------|------|
| **工厂模式** | LLMFactory 统一创建逻辑 |
| **抽象基类** | BaseLLM 定义统一接口 |
| **适配器模式** | 各实现类转换 API 格式 |
| **数据模型** | Pydantic 统一响应格式 |
| **依赖注入** | Agent 接收 BaseLLM 而非具体类 |
| **路由设计** | 根据任务选择合适模型 |

**下一步**: L8 将学习错误处理与重试机制！